# Re-ID evaluation: OSNet on Market-1501 and MSMT17

Reproduce the standard **same-domain** benchmark numbers for OSNet x1.0. Each dataset is scored with the checkpoint trained on that same dataset ([torchreid model zoo](https://kaiyangzhou.github.io/deep-person-reid/MODEL_ZOO)). That's the fair comparison against published results. Section 4 adds a small MSMT17 gallery retrieval demo (top-k crops with green/red borders).

| Dataset | Size | T4 runtime | Expected (euclidean) |
|---|---|---|---|
| Market-1501 | ~1.3 GB | ~10 min | R1 ~94.2 / mAP ~82.6 |
| MSMT17 | ~4 GB | ~40 min | R1 ~74.9 / mAP ~43.8 |

> **Why not the default checkpoint?** `ReIDModel.from_pretrained()` ships OSNet trained on MSMT17 with `combineall=True` (train+test combined). Evaluating that on MSMT17 leaks the test set (near-100%) and on Market-1501 only measures cross-domain transfer (~61 R1). Below we load each dataset's own same-domain checkpoint instead.

> **Runtime:** T4 GPU (`Runtime → Change runtime type`).

---

## 1. Setup

Install **re-ID** from PyPI (`import reid`). Colab already ships CUDA PyTorch, which pip will reuse when it satisfies the package requirement.


In [ ]:
!pip install -q --upgrade pip
!pip install -q reid matplotlib scikit-learn

In [ ]:
import os
import zipfile
from pathlib import Path

import gdown
import torch

from reid import ReIDEvaluator, ReIDModel

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
    ROOT = Path("/content")
except ImportError:
    IN_COLAB = False
    ROOT = Path("..").resolve()

CKPT_DIR = ROOT / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)

device = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | {device}")


def download_checkpoint(gdrive_id: str, filename: str) -> str:
    """Download an OSNet checkpoint from the torchreid model zoo (Google Drive)."""
    path = CKPT_DIR / filename
    if not path.exists():
        gdown.download(id=gdrive_id, output=str(path), quiet=False)
    return str(path)


def print_comparison(name, cos, euc, zoo_r1, zoo_map):
    """Print a cosine-vs-euclidean comparison table against model-zoo targets."""
    print(f"\n{name}: distance metric comparison")
    print(f"{'metric':<10}{'cosine':>10}{'euclidean':>12}{'model zoo':>12}")
    print("-" * 44)
    print(f"{'Rank-1':<10}{cos.rank1:>9.1f}%{euc.rank1:>11.1f}%{zoo_r1:>11.1f}%")
    print(f"{'mAP':<10}{cos.mean_average_precision:>9.1f}%{euc.mean_average_precision:>11.1f}%{zoo_map:>11.1f}%")
    print(f"{'Rank-5':<10}{cos.rank5:>9.1f}%{euc.rank5:>11.1f}%{'-':>12}")
    print(f"{'Rank-10':<10}{cos.rank10:>9.1f}%{euc.rank10:>11.1f}%{'-':>12}")
    print(f"{'mINP':<10}{cos.minp:>9.1f}%{euc.minp:>11.1f}%{'-':>12}")

---
## 2. Market-1501

**Expected (OSNet x1.0, Market-trained, model zoo):** R1 ≈ 94.2 / mAP ≈ 82.6

The model-zoo protocol uses raw **euclidean** distance. We also report **cosine** (L2-normalised embeddings, our default) so you can see the gap.

Market-1501 is a public research benchmark. This notebook downloads the public Google Drive archive linked by the [dataset authors](https://zheng-lab-anu.github.io/Project/project_reid.html) (~1.3 GB extracted); no Roboflow-private data or credentials are used. Review the dataset's terms before use. If Google Drive fails, use the fallback cell below.


In [ ]:
MARKET_ZIP = "/content/Market-1501.zip"
MARKET_DIR = "/content/Market-1501-v15.09.15"

if not os.path.exists(MARKET_DIR):
    # Google Drive file ID for Market-1501-v15.09.15.zip
    gdown.download(id="0B8-rUzbwVRk0c054eEozWG9COHM", output=MARKET_ZIP, quiet=False)
    with zipfile.ZipFile(MARKET_ZIP, "r") as zf:
        zf.extractall("/content")
    print("Extracted to", MARKET_DIR)
else:
    print("Already downloaded.")

In [ ]:
# Fallback if gdown fails: paste your mirror URL:
# !wget -q -O /content/Market-1501.zip "<YOUR_MIRROR_URL>"
# !unzip -q /content/Market-1501.zip -d /content


In [ ]:
from reid import load_market1501

query_m, gallery_m = load_market1501(MARKET_DIR)
print(f"Market-1501: {len(query_m):,} query, {len(gallery_m):,} gallery")

In [ ]:
# OSNet x1.0 checkpoint trained on Market-1501 (model zoo).
market_ckpt = download_checkpoint("1vduhq5DpN2q1g4fYEZfPI17MJeh9qyrA", "osnet_x1_0_market1501.pth")
model_market = ReIDModel.from_pretrained(market_ckpt, architecture="osnet_x1_0")
evaluator_market = ReIDEvaluator(model_market, batch_size=256)

# Cosine (default): extract embeddings once.
result_market = evaluator_market.evaluate(query_m, gallery_m, distance="cosine", return_distmat=False)
# Euclidean (model-zoo protocol): re-score cached embeddings.
result_market_euc = evaluator_market.evaluate(
    query_m,
    gallery_m,
    distance="euclidean",
    return_distmat=False,
    query_embeddings=result_market.query_embeddings,
    gallery_embeddings=result_market.gallery_embeddings,
)

print_comparison("Market-1501", result_market.metrics, result_market_euc.metrics, 94.2, 82.6)

---
## 3. MSMT17

**Expected (OSNet x1.0, MSMT-trained, model zoo):** R1 ≈ 74.9 / mAP ≈ 43.8 (euclidean; cosine also printed)

MSMT17 is distributed by Peking University NELVT for academic use only. Request the official **MSMT17_V1** release after signing the [MSMT17 release agreement](https://www.pkuvmc.com/agreement/RELEASE_AGREEMENT-MSMT17.pdf). Do not use unofficial re-uploads until the archive can be verified against the official distribution.

After download, point `MSMT17_DIR` at the extracted root (must contain `test/`, `list_query.txt`, and `list_gallery.txt`).


In [ ]:
# Official MSMT17_V1 only: request from NELVT after signing the release agreement.
# https://www.pkuvmc.com/agreement/RELEASE_AGREEMENT-MSMT17.pdf
MSMT17_DIR = "/content/MSMT17_V1"  # extracted official root

assert os.path.isdir(MSMT17_DIR), (
    f"MSMT17 not found at {MSMT17_DIR}. "
    "Request the official MSMT17_V1 release, extract it, then set MSMT17_DIR."
)
assert os.path.isfile(os.path.join(MSMT17_DIR, "list_query.txt")), "Missing list_query.txt"
assert os.path.isfile(os.path.join(MSMT17_DIR, "list_gallery.txt")), "Missing list_gallery.txt"
assert os.path.isdir(os.path.join(MSMT17_DIR, "test")), "Missing test/"
print("Using official MSMT17 at", MSMT17_DIR)


In [ ]:
from reid import load_msmt17

query_ms, gallery_ms = load_msmt17(MSMT17_DIR)
print(f"MSMT17: {len(query_ms):,} query, {len(gallery_ms):,} gallery")

In [ ]:
# Same-domain MSMT17 checkpoint. Skip the default combineall alias (test leakage).
msmt17_ckpt = download_checkpoint("112EMUfBPYeYg70w-syK6V6Mx8-Qb9Q1M", "osnet_x1_0_msmt17.pth")
model_msmt17 = ReIDModel.from_pretrained(msmt17_ckpt, architecture="osnet_x1_0")
evaluator_msmt17 = ReIDEvaluator(model_msmt17, batch_size=256)

# Cosine (default): extract embeddings once.
result_msmt17 = evaluator_msmt17.evaluate(query_ms, gallery_ms, distance="cosine", return_distmat=False)
# Euclidean (model-zoo protocol): re-score cached embeddings.
result_msmt17_euc = evaluator_msmt17.evaluate(
    query_ms,
    gallery_ms,
    distance="euclidean",
    return_distmat=False,
    query_embeddings=result_msmt17.query_embeddings,
    gallery_embeddings=result_msmt17.gallery_embeddings,
)

print_comparison("MSMT17", result_msmt17.metrics, result_msmt17_euc.metrics, 74.9, 43.8)

---
## 4. Gallery retrieval demo (MSMT17)

Repeat for **5** MSMT17 queries: each row is one query plus its top-k gallery hits for the same-domain OSNet encoder. Borders are **green** when the gallery crop is the same identity and **red** otherwise. Ranking uses cosine distance on the embeddings from section 3 (no re-extract), with the usual same-camera / junk-ID filter.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageOps

TOP_K = 10
NUM_QUERIES = 5
SEED = 0
# Optional pin list, e.g. [12, 40, 100]. None samples NUM_QUERIES eligible queries (distinct identities).
QUERY_INDICES = None


def cosine_distmat(query_embeddings: np.ndarray, gallery_embeddings: np.ndarray) -> np.ndarray:
    q = query_embeddings / (np.linalg.norm(query_embeddings, axis=1, keepdims=True) + 1e-12)
    g = gallery_embeddings / (np.linalg.norm(gallery_embeddings, axis=1, keepdims=True) + 1e-12)
    return (1.0 - (q @ g.T)).astype(np.float32)


def rank_gallery(q_idx: int, query, gallery, distmat: np.ndarray, top_k: int) -> list[int]:
    order = np.argsort(distmat[q_idx])
    q_pid = int(query.pids[q_idx])
    q_cam = int(query.camids[q_idx])
    junk_pids = gallery.gallery_junk_pids
    ranked: list[int] = []
    for g_idx in order:
        g_pid = int(gallery.pids[g_idx])
        g_cam = int(gallery.camids[g_idx])
        if g_pid in junk_pids:
            continue
        if g_pid == q_pid and g_cam == q_cam:
            continue
        ranked.append(int(g_idx))
        if len(ranked) >= top_k:
            break
    return ranked


def has_valid_match(q_idx: int, query, gallery) -> bool:
    q_pid = int(query.pids[q_idx])
    q_cam = int(query.camids[q_idx])
    junk_pids = gallery.gallery_junk_pids
    for g_idx in range(len(gallery)):
        g_pid = int(gallery.pids[g_idx])
        g_cam = int(gallery.camids[g_idx])
        if g_pid in junk_pids:
            continue
        if g_pid == q_pid and g_cam != q_cam:
            return True
    return False


def border_image(path: str, color: str, width: int = 6) -> Image.Image:
    return ImageOps.expand(Image.open(path).convert("RGB"), border=width, fill=color)


def sample_query_indices(query, gallery, num_queries: int, seed: int) -> list[int]:
    eligible = [i for i in range(len(query)) if has_valid_match(i, query, gallery)]
    if not eligible:
        raise RuntimeError("No MSMT17 query has a valid cross-camera gallery match.")

    rng = np.random.default_rng(seed)
    rng.shuffle(eligible)
    chosen: list[int] = []
    seen_pids: set[int] = set()
    for q_idx in eligible:
        pid = int(query.pids[q_idx])
        if pid in seen_pids:
            continue
        chosen.append(q_idx)
        seen_pids.add(pid)
        if len(chosen) >= num_queries:
            break

    if len(chosen) < num_queries:
        raise RuntimeError(
            f"Only found {len(chosen)} eligible identities with valid matches; need {num_queries}."
        )
    return chosen


distmat_ms = cosine_distmat(result_msmt17.query_embeddings, result_msmt17.gallery_embeddings)
query_indices = list(QUERY_INDICES) if QUERY_INDICES is not None else sample_query_indices(
    query_ms, gallery_ms, NUM_QUERIES, SEED
)

n_cols = 1 + TOP_K
fig, axes = plt.subplots(
    len(query_indices),
    n_cols,
    figsize=(2.0 * n_cols, 3.0 * len(query_indices)),
)
if len(query_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, q_idx in enumerate(query_indices):
    ranked = rank_gallery(q_idx, query_ms, gallery_ms, distmat_ms, TOP_K)
    q_pid = int(query_ms.pids[q_idx])
    q_cam = int(query_ms.camids[q_idx])
    q_path = query_ms.image_paths[q_idx]

    axes[row, 0].imshow(border_image(q_path, "#1f77b4"))
    axes[row, 0].set_title(f"query {q_idx}\npid={q_pid} cam={q_cam}", fontsize=8)
    axes[row, 0].axis("off")

    hits = 0
    for col, g_idx in enumerate(ranked, start=1):
        g_pid = int(gallery_ms.pids[g_idx])
        g_cam = int(gallery_ms.camids[g_idx])
        correct = g_pid == q_pid
        hits += int(correct)
        color = "#2ca02c" if correct else "#d62728"
        axes[row, col].imshow(border_image(gallery_ms.image_paths[g_idx], color))
        axes[row, col].set_title(
            f"#{col} d={distmat_ms[q_idx, g_idx]:.3f}\npid={g_pid} cam={g_cam}",
            fontsize=8,
            color=color,
        )
        axes[row, col].axis("off")

    for col in range(1 + len(ranked), n_cols):
        axes[row, col].axis("off")

    print(
        f"Query {q_idx} ({Path(q_path).name}): "
        f"correct in top-{TOP_K} = {hits}/{len(ranked)}"
    )

fig.suptitle(
    f"MSMT17 retrieval  |  {len(query_indices)} queries  |  top-{TOP_K}",
    y=1.01,
)
plt.tight_layout()
plt.show()


---
## 5. Summary


In [ ]:
print(f"{'Dataset':<14}{'encoder':<22}{'distance':<12}{'mAP':>8}{'Rank-1':>9}{'Rank-5':>9}{'Rank-10':>9}{'mINP':>8}")
print("-" * 91)


def _row(name, encoder, dist, m):
    return (
        f"{name:<14}{encoder:<22}{dist:<12}{m.mean_average_precision:>7.1f}%{m.rank1:>8.1f}%"
        f"{m.rank5:>8.1f}%{m.rank10:>8.1f}%{m.minp:>7.1f}%"
    )


print(_row("Market-1501", "OSNet", "cosine", result_market.metrics))
print(_row("", "", "euclidean", result_market_euc.metrics))
print(_row("MSMT17", "OSNet", "cosine", result_msmt17.metrics))
print(_row("", "", "euclidean", result_msmt17_euc.metrics))
print("-" * 91)
print("Model-zoo targets (OSNet, euclidean): Market-1501 R1≈94.2 mAP≈82.6  |  MSMT17 R1≈74.9 mAP≈43.8")